# EKF · Flujo medio e instantáneo

El caso de referencia reproduce el EKF cinético del notebook 09. En cada cierre de 30 minutos asimila el flujo medio a través del acumulador \(I\). El segundo filtro propaga seis estados físicos y asimila el último dato bruto de flujo disponible en esa ventana, comparándolo con la emisión predicha al cierre. La tabla registra la antigüedad del dato. Los dos casos usan el mismo \(R=0{,}00218\ (\mathrm{g/(L\cdot h)})^2\), los mismos horarios y criterios de actualización, los mismos coeficientes nominales y la misma \(D_\theta\). Esta comparación mantiene fijo \(R\) según el diseño del ensayo. Una muestra instantánea y un promedio pueden tener distinta incertidumbre de medición.

Al cierre de la ventana, el filtro instantáneo propaga la covarianza conjunta de los seis estados y del flujo evaluado allí. Las diferencias finitas incluyen el efecto directo de las tasas cinéticas en ese flujo, además de su efecto a través de los estados. El flujo es una salida algebraica del modelo y no se guarda como estado. La integral de flujo que se calcula aparte para puntuar la predicción media es un diagnóstico nominal. Su valor y su covarianza no participan en la actualización del filtro instantáneo.

Se informa el RMSE de ambos filtros contra las mismas referencias de flujo y contra las mismas muestras químicas. El NIS de cada caso corresponde a la observación que asimila, por lo que se interpreta con esa diferencia de señal.

In [ ]:
from pathlib import Path
import ast
import json
import os
import sys
from types import ModuleType

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/wine_fermentation").is_dir())
EKF = ROOT / "notebooks/EKF"
HERE = EKF / "kalman_analysis"
prefix = Path(os.environ["CONDA_PREFIX"]).resolve()
assert prefix.name.lower() == "fermentation" and Path(sys.prefix).resolve() == prefix
print(f"Kernel {prefix.name} · SciPy {scipy.__version__}")

# Load the existing helpers as Python functions. The filter below uses the
# vectorized NumPy transition from EKF05 and does not require Numba.
core_path = HERE / "analysis_core.py"
tree = ast.parse(core_path.read_text(encoding="utf-8"))
tree.body = [node for node in tree.body
             if not (isinstance(node, ast.ImportFrom) and node.module == "numba")]
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        node.decorator_list = [
            d for d in node.decorator_list
            if not (isinstance(d, ast.Call) and isinstance(d.func, ast.Name) and d.func.id == "njit")
        ]
core = ModuleType("analysis_core")
core.__file__ = str(core_path)
exec(compile(tree, str(core_path), "exec"), vars(core))

# Reuse the six-lot input preparation from the four-source comparison.
reference = HERE / "04_gf_variable_solubility_fourth_noise_source.source.ipynb"
reference_nb = nbformat.read(reference, as_version=4)
preparation = next(cell.source for cell in reference_nb.cells
                   if cell.cell_type == "code" and "DATA,CHEM,AUDIT,_=core.prepare(ROOT)" in cell.source)
assert preparation.count("import analysis_core as core") == 1
preparation = preparation.replace("import analysis_core as core", "")
exec(compile(preparation.replace("display(AUDIT)", ""), str(reference), "exec"))

calibration = ROOT / "notebooks/diagnostics/zenteno_lab004_009_incertidumbre_exploratoria.ipynb"
saved_nb = nbformat.read(calibration, as_version=4)
saved = json.loads(saved_nb.cells[-1].outputs[-1].text)
rate_names = ["mu0", "betaG0", "betaF0"]
assert saved["orden_D_theta"] == rate_names
D_theta = np.array(saved["D_theta_Yef_fijo"], dtype=float)
np.testing.assert_array_equal(D_theta, np.diag(np.diag(D_theta)))
assert (np.diag(D_theta) > 0).all()
R = 0.00218
R_cell = next(cell.source for cell in reference_nb.cells
              if cell.cell_type == "code" and "R = 0.00218" in cell.source)
assert ast.literal_eval(ast.parse(R_cell).body[0].value) == R
theta = core.THETA
nominal = np.array([theta[name] for name in rate_names])
sigma_theta = np.sqrt(np.diag(D_theta))
SCALE = core.SCALE
scale_outer = np.outer(SCALE, SCALE)
KLA, GAIN = core.ARCHIVED_GAS
PCO2_BAR = 1.0


In [ ]:
# Reuse the equations from the vectorized G/F EKF. Supply one set of three
# rate coefficients per trajectory, with the other coefficients shared.
source05 = EKF / "05_zenteno_co2_variable_solubility.source.ipynb"
nb05 = nbformat.read(source05, as_version=4)
functions = {}
for cell in nb05.cells:
    if cell.cell_type == "code":
        for node in ast.parse(cell.source).body:
            if isinstance(node, ast.FunctionDef):
                functions[node.name] = ast.get_source_segment(cell.source, node)
for name in ("co2_saturation", "gas_exchange"):
    exec(compile(functions[name], str(source05), "exec"))

rates_source = functions["rates_ekf"]
for original, replacement in (
    ("def rates_ekf(w,t,T,S0):", "def rates_ekf(w,t,T,S0,rate_coefficients):"),
    ("theta['mu0']", "rate_coefficients[:, 0]"),
    ("theta['betaG0']", "rate_coefficients[:, 1]"),
    ("theta['betaF0']", "rate_coefficients[:, 2]"),
):
    assert rates_source.count(original) == 1
    rates_source = rates_source.replace(original, replacement)
exec(compile(rates_source, str(source05), "exec"))


def transition(points, rate_coefficients, start, end, data):
    """Integrate each trajectory with its own constant rates over this interval."""
    w = points * SCALE
    n = max(1, int(np.ceil((end - start) * 60 - 1e-7)))
    dt = (end - start) / n
    available = data["available"]
    temperatures = data["temperature"]
    for j in range(n):
        t = start + j * dt
        times = np.array([t, t + dt / 2, t + dt])
        indices = np.searchsorted(available, times + 1e-12, side="right") - 1
        if indices[0] < 0:
            raise ValueError("No hay temperatura causal al comienzo del intervalo")
        T0, Tm, T1 = temperatures[indices]
        S0 = data["S0"]
        k1 = rates_ekf(w, t, T0, S0, rate_coefficients)
        k2 = rates_ekf(w + dt * k1 / 2, t + dt / 2, Tm, S0, rate_coefficients)
        k3 = rates_ekf(w + dt * k2 / 2, t + dt / 2, Tm, S0, rate_coefficients)
        k4 = rates_ekf(w + dt * k3, t + dt, T1, S0, rate_coefficients)
        w = np.maximum(w + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6, 0.)
    return w / SCALE


def sensitivities(z, start, end, data, kinetic, relative_step=1e-4):
    state_step = 1e-5
    delta = np.eye(len(z)) * state_step
    lower = z - delta * (z >= state_step)[:, None]
    denominator = state_step * (1 + (z >= state_step))
    points = np.vstack([z, z + delta, lower])
    coefficients = np.tile(nominal, (len(points), 1))
    if kinetic:
        parameter_step = relative_step * nominal
        parameter_delta = np.diag(parameter_step)
        points = np.vstack([points, np.tile(z, (6, 1))])
        coefficients = np.vstack([coefficients, nominal + parameter_delta, nominal - parameter_delta])

    predicted = transition(points, coefficients, start, end, data)
    J = ((predicted[1:8] - predicted[8:15]) / denominator[:, None]).T
    G = ((predicted[15:18] - predicted[18:21]) / (2 * parameter_step[:, None])).T if kinetic else None
    return predicted[0], J, G


# Compare with the archived scalar RHS and transition in several regimes.
data = DATA["LAB019"]
for start, state in (
    (3., [0.4, .20, 90., 95., 15., .02, 0.]),
    (31., [1.8, .12, 65., 70., 36., .65, .01]),
    (75., [2.4, .03, 27., 43., 65., 1.0, .02]),
    (120., [.9, 0., 0., 12., 94., 1.2, .03]),
):
    w = np.asarray(state)
    z = w / SCALE
    temp = core.temperature_at(start, data["available"], data["temperature"])
    np.testing.assert_allclose(
        rates_ekf(w[None, :], start, temp, data["S0"], nominal[None, :])[0],
        core.rhs(w, temp, data["S0"], KLA, GAIN), atol=1e-13, rtol=1e-12,
    )
    end = start + 1 / 12
    zp, J, G = sensitivities(z, start, end, data, True)
    z_ref, J_ref = core.jacobian(z, start, end, data["available"], data["temperature"],
                                 data["S0"], KLA, GAIN, 1 / 60)
    np.testing.assert_allclose(zp, z_ref, atol=1e-13)
    np.testing.assert_allclose(J, J_ref, atol=1e-9)
    _, _, G_fine = sensitivities(z, start, end, data, True, relative_step=5e-5)
    np.testing.assert_allclose(G, G_fine, atol=5e-7, rtol=3e-3)
print("Verificadas la dinámica nominal, las sensibilidades y el paso de diferencias finitas.")


In [ ]:
# Preserve the raw sources, conversion and clocks of the existing windows.
recent = core.recent_raw(
    ROOT / "data/original/natural/lab019_021/backup_global_co2_completo.csv",
    ["timestamp", "fermentador", "flow_sccm"],
)
flow_factor = 44.0095 * 60 * INSTRUMENT["co2_correction_factor"] / (1000 * 24.16)
instant_measurements = {}
age_rows = []
for i, lab in enumerate(LABS):
    d = DATA[lab]
    if lab in DEVELOPMENT:
        raw = recent.loc[recent.fermentador.eq(f"F{i + 1}")].copy()
    else:
        raw = v.raw_co2[lab].copy()
    raw = raw.dropna(subset=["flow_sccm"]).sort_values("t_h")
    withdrawals = volume_rows.loc[volume_rows.lab.eq(lab) &
                                  volume_rows.record_kind.eq("withdrawal_after")].sort_values("time_h")
    sample_times = raw.t_h.to_numpy()
    remaining = np.r_[2., withdrawals.remaining_volume_L][
        np.searchsorted(withdrawals.time_h, sample_times, side="right")
    ]
    raw["converted"] = raw.flow_sccm.to_numpy() * flow_factor / remaining
    raw["bin"] = np.ceil(sample_times / .5 - 1e-10).astype(int)
    means = raw.groupby("bin").converted.mean()
    last = raw.drop_duplicates("bin", keep="last").set_index("bin")
    grid = d["grid"]
    indices = np.flatnonzero(d["valid"])
    bins = np.rint(grid[indices] / .5).astype(int)
    np.testing.assert_allclose(means.reindex(bins), d["y"][indices], atol=1e-12, rtol=0)
    points = last.reindex(bins)
    age_seconds = (grid[indices] - points.t_h.to_numpy()) * 3600
    assert np.isfinite(age_seconds).all() and np.isfinite(points.converted).all()
    assert age_seconds.min() >= -1e-6 and age_seconds.max() <= 120.000001
    observations = np.full(len(grid), np.nan)
    observations[indices] = points.converted
    assert np.isfinite(observations[d["update"]]).all()
    instant_measurements[lab] = observations
    age_rows.append({"LAB": lab, "Ventanas válidas": len(indices), "Actualizaciones": d["update"].sum(),
                     "Edad mediana [s]": np.median(age_seconds), "Edad máxima [s]": np.max(age_seconds)})
display(pd.DataFrame(age_rows).round(2))


In [ ]:
scale6 = SCALE[:6]
scale_outer6 = np.outer(scale6, scale6)

def rate_vector(w, t, temperature, data, coefficients):
    # rates_ekf reads six physical inputs. Its seventh output is emission rate.
    return rates_ekf(w, t, temperature, data["S0"], coefficients)


def transition6(points, coefficients, start, end, data):
    """Six-state RK4 and nominal integral used only for common-target scoring."""
    w = points * scale6
    n = max(1, int(np.ceil((end - start) * 60 - 1e-7)))
    dt = (end - start) / n
    emitted = 0.
    for j in range(n):
        t = start + j * dt
        times = np.array([t, t + dt / 2, t + dt])
        indices = np.searchsorted(data["available"], times + 1e-12, side="right") - 1
        assert indices[0] >= 0
        T0, Tm, T1 = data["temperature"][indices]
        k1 = rate_vector(w, t, T0, data, coefficients)
        k2 = rate_vector(w + dt * k1[:, :6] / 2, times[1], Tm, data, coefficients)
        k3 = rate_vector(w + dt * k2[:, :6] / 2, times[1], Tm, data, coefficients)
        k4 = rate_vector(w + dt * k3[:, :6], times[2], T1, data, coefficients)
        increment = dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        w = np.maximum(w + increment[:, :6], 0.)
        emitted += increment[0, 6]
    return w / scale6, emitted


def emission(points, coefficients, time, data):
    T = core.temperature_at(time, data["available"], data["temperature"])
    return rate_vector(points * scale6, time, T, data, coefficients)[:, 6]


def linearization6(z, start, end, data, dose, at_close):
    n = len(z)
    epsilon = 1e-5
    delta = np.eye(n) * epsilon
    lower = z - delta * (z >= epsilon)[:, None]
    denominator = epsilon * (1 + (z >= epsilon))
    theta_step = 1e-4 * nominal
    points = np.vstack([z, z + delta, lower, np.tile(z, (6, 1))])
    coefficients = np.vstack([np.tile(nominal, (1 + 2 * n, 1)),
                              nominal + np.diag(theta_step), nominal - np.diag(theta_step)])
    predicted, emitted = transition6(points, coefficients, start, end, data)
    predicted[:, 1] += dose / scale6[1]
    if at_close:
        predicted = np.column_stack([predicted, emission(predicted, coefficients, end, data)])
    J = ((predicted[1:n+1] - predicted[n+1:2*n+1]) / denominator[:, None]).T
    G = ((predicted[2*n+1:2*n+4] - predicted[2*n+4:]) / (2 * theta_step[:, None])).T
    return predicted[0], J, G, emitted


# Without conditioning on flow, the six-state propagation is identical.
d = DATA["LAB019"]
test = np.array([1.2, .11, 53., 56., 42., .6]) / scale6
end = 35 + 1/12
six, J6, G6, integral = linearization6(test, 35., end, d, 0., True)
seven, J7, G7 = sensitivities(np.r_[test, 0.], 35., end, d, True)
np.testing.assert_allclose(six[:6], seven[:6], atol=1e-13)
np.testing.assert_allclose(J6[:6], J7[:6, :6], atol=1e-9)
np.testing.assert_allclose(G6[:6], G7[:6], atol=1e-9)
np.testing.assert_allclose(integral, seven[6] * SCALE[6], atol=1e-13)
np.testing.assert_allclose(six[6], emission(seven[None, :6], nominal[None, :], end, d)[0])


In [ ]:
flux_variances = np.square(core.BASE_SIGMA)

def replay(data, kinetic):
    grid = data["grid"]
    z = data["initial"] / SCALE
    P = data["P0"] / scale_outer
    states = np.empty((len(grid), len(z)))
    covariance = np.empty((len(grid), len(z), len(z)))
    diag = np.full((len(grid), 7), np.nan)
    injected = np.zeros_like(covariance)
    for i, end in enumerate(grid):
        if i:
            start = grid[i - 1]
            before = z
            z, J, G = sensitivities(z, start, end, data, kinetic)
            if kinetic:
                # G already differentiates the integrated transition.
                scaled_G = G * sigma_theta
                Q = scaled_G @ scaled_G.T
            else:
                T = core.temperature_at(start, data["available"], data["temperature"])
                L = core.noise_L(before * SCALE, T, data["S0"], KLA, GAIN)
                Q = (end - start) * ((L * flux_variances) @ L.T) / scale_outer
            injected[i] = Q * scale_outer
            P = J @ P @ J.T + Q
            P = (P + P.T) / 2

        z[1] += data["doses"][i] / SCALE[1]
        if end > 0 and abs(end / .5 - round(end / .5)) < 1e-8:
            predicted = z[6] / .5
            innovation = data["y"][i] - predicted
            S = 4 * P[6, 6] + R
            clips = 0
            if data["update"][i]:
                K = 2 * P[:, 6] / S
                updated = z + K * innovation
                clips = np.count_nonzero(updated[:6] < 0)
                B = np.eye(len(z))
                B[:, 6] -= 2 * K
                P = B @ P @ B.T + np.outer(K, K) * R
                P = (P + P.T) / 2
                z = np.maximum(updated, 0)
            diag[i] = [predicted, innovation, S, innovation**2 / S, clips,
                       data["update"][i], data["valid"][i]]
            z[6] = 0.
            P[6, :] = P[:, 6] = 0.
        states[i] = z * SCALE
        covariance[i] = P * scale_outer
    assert np.isfinite(states).all() and np.isfinite(covariance).all()
    assert np.linalg.eigvalsh(injected).min() > -1e-9
    np.testing.assert_array_equal(diag[:, 5] == 1, data["update"])
    return (states, covariance, diag), injected

def replay_instant(data, observed):
    grid = data["grid"]
    z = data["initial"][:6] / scale6
    P = data["P0"][:6, :6] / scale_outer6
    states = np.empty((len(grid), 6))
    covariance = np.empty((len(grid), 6, 6))
    diag = np.full((len(grid), 7), np.nan)
    average = np.full(len(grid), np.nan)
    emitted = 0.
    for i, end in enumerate(grid):
        if i:
            at_close = end > 0 and abs(end / .5 - round(end / .5)) < 1e-8
            forecast, J, G, increment = linearization6(
                z, grid[i - 1], end, data, data["doses"][i], at_close
            )
            emitted += increment
            scaled_G = G * sigma_theta
            joint = J @ P @ J.T + scaled_G @ scaled_G.T
            joint = (joint + joint.T) / 2
            z = forecast[:6]
            P = joint[:6, :6]
            if at_close:
                predicted = forecast[6]
                average[i] = emitted / .5
                emitted = 0.
                innovation = observed[i] - predicted
                S = joint[6, 6] + R
                clips = 0
                if data["update"][i]:
                    K = joint[:6, 6] / S
                    corrected = z + K * innovation
                    clips = np.count_nonzero(corrected < 0)
                    z = np.maximum(corrected, 0.)
                    # Joseph form on the joint prediction (states and output).
                    projection = np.column_stack([np.eye(6), -K])
                    P = projection @ joint @ projection.T + R * np.outer(K, K)
                    P = (P + P.T) / 2
                diag[i] = [predicted, innovation, S, innovation**2 / S, clips,
                           data["update"][i], data["valid"][i]]
        else:
            z[1] += data["doses"][i] / scale6[1]
        states[i] = z * scale6
        covariance[i] = P * scale_outer6
    assert np.isfinite(states).all() and np.isfinite(covariance).all()
    assert np.linalg.eigvalsh(covariance).min() > -1e-9
    assert (diag[np.isfinite(diag[:, 0]), 2] >= R - 1e-12).all()
    np.testing.assert_array_equal(diag[:, 5] == 1, data["update"])
    return states, covariance, diag, average


def instant_from_reference(data, frame, diag):
    prediction = np.full(len(data["grid"]), np.nan)
    for i in np.flatnonzero(np.isfinite(diag[:, 0])):
        start, end = data["grid"][i-1:i+1]
        z, _ = transition6(frame[i-1:i, :6] / scale6, nominal[None, :], start, end, data)
        z[0, 1] += data["doses"][i] / scale6[1]
        prediction[i] = emission(z, nominal[None, :], end, data)[0]
    return prediction


MEAN = "Con I"
INSTANT = "Sin I"
results = {MEAN: {}, INSTANT: {}}
flow = {}
state_rows, nis_rows = [], []
for lab in LABS:
    print(lab, flush=True)
    d = DATA[lab]
    y_instant = instant_measurements[lab]
    base, _ = replay(d, kinetic=True)
    candidate_state, candidate_cov, candidate_diag, candidate_average = replay_instant(d, y_instant)
    candidate = (candidate_state, candidate_cov, candidate_diag)
    results[MEAN][lab], results[INSTANT][lab] = base, candidate
    flow[lab] = {
        "Promedio 30 min": (d["y"], base[2][:, 0], candidate_average),
        "Instantáneo": (y_instant, instant_from_reference(d, base[0], base[2]), candidate_diag[:, 0]),
    }
    for case, result in ((MEAN, base), (INSTANT, candidate)):
        scores, _, _ = core.diagnostics(DATA, CHEM, lab, result)
        state_rows.append(scores.loc[scores.variable.ne("CO2"), ["LAB", "variable", "unit", "n", "RMSE"]]
                          .assign(Grupo=ROLE[lab], EKF=case))
        diag = result[2]
        m = d["update"]
        np.testing.assert_array_equal(diag[:, 5] == 1, m)
        nis_rows.append(dict(LAB=lab, EKF=case, NIS=diag[m, 3].mean(),
                             Recortes=int(diag[m, 4].sum()), Actualizaciones=int(m.sum())))

flow_rows = []
for lab, targets in flow.items():
    mask = DATA[lab]["valid"]
    for target, (observed, with_I, without_I) in targets.items():
        for case, predicted in ((MEAN, with_I), (INSTANT, without_I)):
            error = predicted[mask] - observed[mask]
            assert np.isfinite(error).all()
            flow_rows.append(dict(Grupo=ROLE[lab], LAB=lab, Referencia=target, EKF=case,
                                  n=int(mask.sum()), RMSE=np.sqrt(np.mean(error**2))))
flow_scores = pd.DataFrame(flow_rows)
state_scores = pd.concat(state_rows, ignore_index=True)
flow_table = flow_scores.pivot(index=["Grupo", "LAB", "Referencia", "n"], columns="EKF", values="RMSE")
state_table = state_scores.pivot(index=["Grupo", "LAB", "variable", "unit", "n"], columns="EKF", values="RMSE")
assert flow_table.notna().all().all() and state_table.notna().all().all()
# The mean flow benchmark matches the executed notebook 09.
baseline = flow_scores.loc[flow_scores.EKF.eq(MEAN) & flow_scores.Referencia.eq("Promedio 30 min")]
np.testing.assert_allclose(baseline.RMSE, [.06539, .06233, .08278, .06754, .07849, .07558], atol=5.1e-6)
print("RMSE de flujo [g/(L h)] con referencia común, promedio por lote")
display(flow_table.groupby(level=["Grupo", "Referencia"]).mean().round(5))
print("RMSE de estados, promedio por lote")
display(state_table.groupby(level=["Grupo", "variable", "unit"]).mean().round(5))
print("RMSE de flujo por lote")
display(flow_table.round(5))
print("RMSE de estados por lote")
display(state_table.round(5))
print("NIS sobre la señal asimilada por cada filtro")
display(pd.DataFrame(nis_rows).round(4))


In [ ]:
colors = {MEAN: "#0072B2", INSTANT: "#D55E00"}
styles = {MEAN: "-", INSTANT: "--"}

def series(frame, name):
    if name == "GF":
        return frame[:, 2] + frame[:, 3]
    return frame[:, core.STATES.index(name)]


def plot_panel(variable, title, unit, target=None):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, lab in zip(axes.flat, LABS):
        d = DATA[lab]
        time = d["grid"]
        factor = 1000 if variable == "N" else 1
        if target is not None:
            mask = d["valid"]
            observed, with_I, without_I = flow[lab][target]
            ax.plot(time[mask], observed[mask], c="black", lw=1, label="Samples")
            for case, predicted in ((MEAN, with_I), (INSTANT, without_I)):
                ax.plot(time[mask], predicted[mask], color=colors[case], ls=styles[case], lw=1.5, label=case)
            scores = flow_scores.loc[flow_scores.LAB.eq(lab) & flow_scores.Referencia.eq(target)]
        else:
            for case in (MEAN, INSTANT):
                frame = results[case][lab][0]
                ax.plot(time, factor * series(frame, variable),
                        color=colors[case], ls=styles[case], lw=1.5, label=case)
            if variable in core.CHEM_COLS:
                col = core.CHEM_COLS[variable]
                samples = CHEM.loc[CHEM.lab.eq(lab) & CHEM.t_h.gt(0) &
                                   CHEM.t_h.le(d["horizon"])].dropna(subset=[col])
                if variable == "GF":
                    samples = samples.loc[~samples.GF_zero_substitution.astype(str).str.lower().eq("true")]
                initial = series(d["initial"][None, :], variable)[0]
                ax.scatter(np.r_[0., samples.t_h], factor * np.r_[initial, samples[col]],
                           c="black", s=23, label="Samples", zorder=5)
            scores = state_scores.loc[state_scores.LAB.eq(lab) & state_scores.variable.eq(variable)]

        ax.set_ylim(bottom=0.)
        if variable == "GF":
            density = DENSITY.loc[DENSITY.lab.eq(lab) &
                                  DENSITY.t_h.between(0, d["horizon"])].dropna(subset=["density_g_l"])
            if len(density):
                values = 2.5 * (density.density_g_l - 1000)
                ax.scatter(density.t_h, values, marker="D", facecolors="none",
                           edgecolors="#D55E00", s=32, label="G+F from density", zorder=4)
                ax.set_ylim(bottom=min(0., float(values.min()) * 1.15))
                ax.axhline(0, c="gray", lw=.6)
        for i, pulse in enumerate(time[d["doses"] > 0]):
            ax.axvline(pulse, c="gray", ls=":", lw=.8, label="N pulse" if i == 0 else "_nolegend_")
        error_label = "\n".join(f"RMSE {r.EKF} = {r.RMSE:.3f} {unit}"
                                for r in scores.itertuples()) or None
        ax.set(title=lab, xlabel="Time [h]", ylabel=f"{title} [{unit}]")
        ax.grid(alpha=.2)
        legend = ax.legend(loc="best", fontsize=8, framealpha=.85, title=error_label, title_fontsize=8)
        if error_label:
            legend.get_title().set_ha("left")
    fig.suptitle(title, y=.99)
    fig.tight_layout(pad=.7, h_pad=1.2, w_pad=.9)
    assert not fig.legends
    plt.show()


plot_panel("CO2", "CO₂ flow · 30-min mean", "g/(L h)", target="Promedio 30 min")
plot_panel("CO2", "CO₂ flow · instantaneous", "g/(L h)", target="Instantáneo")
for variable, title, unit in (
    ("X", "Biomass", "g/L"),
    ("N", "YAN", "mg/L"),
    ("G", "Glucose", "g/L"),
    ("F", "Fructose", "g/L"),
    ("GF", "Glucose + fructose", "g/L"),
    ("E", "Ethanol", "g/L"),
    ("Cd", "Dissolved CO₂", "g/L"),
):
    plot_panel(variable, title, unit)
